# Jednaka dužina poruka i maska dopune

Pre modela, svaka poruka mora da postane niz ID-jeva iste dužine. Kraću poruku dopunjavamo ID-jem `<PAD> = 0`; dužu skraćujemo. Uz niz ID-jeva pravimo masku: `False` označava stvarni token, `True` označava dopunu koju kasniji model treba da ignoriše.

## Kratka poruka

Koristimo isti mali rečnik iz prethodne sveske. Za `Free offer!` prvo dobijamo `[2, 1, 4]`. Ako izaberemo dužinu 6, dodajemo tri nule i tri oznake `True`. To nisu nove reči u poruci.

In [1]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.sequences import DEFAULT_MAX_LENGTH, encode_and_pad
from src.tokenization import tokenize_sms
from src.vocabulary import build_vocabulary

toy_vocabulary = build_vocabulary(["Free prize now!", "Free entry now."])
message = "Free offer!"
input_ids, padding_mask = encode_and_pad(message, toy_vocabulary, max_length=6)
tokens = tokenize_sms(message)
pd.DataFrame({
    "pozicija": range(6),
    "token": tokens + ["<PAD>"] * (6 - len(tokens)),
    "ID": input_ids,
    "padding maska": padding_mask,
})

,pozicija,token,ID,padding maska
0,0,free,2,False
1,1,offer,1,False
2,2,!,4,False
3,3,<PAD>,0,True
4,4,<PAD>,0,True
5,5,<PAD>,0,True


`<UNK> = 1` i `<PAD> = 0` imaju različite uloge. `offer` je stvarni deo poruke koji rečnik ne poznaje, zato maska za njega ostaje `False`. Samo dodate nule imaju `True`.

## Duga poruka

Ako poruka ima više tokena od zadate dužine, zadržavamo prvih `max_length`. Nema dopune, pa su sve vrednosti maske `False`. U ovom kratkom primeru biramo dužinu 5 da se skraćivanje jasno vidi.

In [2]:
long_message = "Free prize now! Free entry now."
short_ids, short_mask = encode_and_pad(long_message, toy_vocabulary, max_length=5)
print("Svi tokeni:", tokenize_sms(long_message))
print("Sačuvani ID-jevi:", short_ids)
print("Maska:", short_mask)

Svi tokeni: ['free', 'prize', 'now', '!', 'free', 'entry', 'now', '.']
Sačuvani ID-jevi: [2, 7, 3, 4, 2]
Maska: [False, False, False, False, False]


## Zašto je početni maksimum 128?

Planirana početna konfiguracija koristi 128 pozicija. Broj nije univerzalno pravilo: pogledajmo dužine samo u našem trening skupu. Podaci za završni test se ovde ne čitaju.

In [3]:
train = pd.read_csv(project_root / "data" / "processed" / "sms_train.csv")
lengths = train["text"].map(lambda text: len(tokenize_sms(text)))
pd.DataFrame({
    "mera": ["broj poruka", "medijana tokena", "95. percentil", "najduža poruka", "poruka dužih od 128"],
    "vrednost": [len(train), int(lengths.median()), int(lengths.quantile(0.95)), int(lengths.max()), int((lengths > DEFAULT_MAX_LENGTH).sum())],
})

,mera,vrednost
0,broj poruka,4127
1,medijana tokena,16
2,95. percentil,43
3,najduža poruka,224
4,poruka dužih od 128,4


Sa dužinom 128 skraćujemo samo mali broj trening poruka, ali svaku kraću dopunjavamo. Funkcija vraća dve usklađene liste iste dužine. U narednoj celini ćemo ih pretvoriti u oblike koje PyTorch model prima; do tada su ovo obične Python liste.